In [ ]:
%cd /kaggle/working
!cp -r /kaggle/input/datasets/anfs2003/ragtruth-lettucedetect/lettucedetect_code/LettuceDetect .
%cd LettuceDetect
!pip install -q -e .

/kaggle/working
/kaggle/working/LettuceDetect
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.4/87.4 kB 6.6 MB/s eta 0:00:00
  Building editable for lettucedetect (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.5.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have ju

In [2]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_WRITE")

In [3]:
%cd /kaggle/working/LettuceDetect
!PYTORCH_ALLOC_CONF=expandable_segments:True python scripts/train.py \
    --ragtruth-path /kaggle/input/datasets/anfs2003/ragtruth-lettucedetect/ragtruth_data.json \
    --model-name answerdotai/ModernBERT-large \
    --output-dir /kaggle/working/lettucedetect_full_large \
    --batch-size 2 --epochs 4 --learning-rate 1e-5 --fp16 \
    --hf-repo anfs4554/lettucedetect-full-large

/kaggle/working/LettuceDetect
config.json: 1.19kB [00:00, 575kB/s]
tokenizer_config.json: 20.8kB [00:00, 47.9MB/s]
tokenizer.json: 2.13MB [00:00, 12.2MB/s]
special_tokens_map.json: 100%|█████████████████| 694/694 [00:00<00:00, 3.18MB/s]
model.safetensors: 100%|████████████████████| 1.58G/1.58G [00:05<00:00, 295MB/s]
Loading weights: 100%|█| 172/172 [00:00<00:00, 1545.35it/s, Materializing param=
ModernBertForTokenClassification LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.

Starting training on cuda
Training samples: 13581, Test samples: 1509


Epoch 1/4
Training:   0%|     

In [4]:
!python scripts/evaluate.py --model_path /kaggle/working/lettucedetect_full_large \
    --data_path /kaggle/input/datasets/anfs2003/ragtruth-lettucedetect/ragtruth_data.json \
    --evaluation_type example_level 2>&1 | tee /kaggle/working/eval_full_example_level.txt
!python scripts/evaluate.py --model_path /kaggle/working/lettucedetect_full_large \
    --data_path /kaggle/input/datasets/anfs2003/ragtruth-lettucedetect/ragtruth_data.json \
    --evaluation_type char_level 2>&1 | tee /kaggle/working/eval_full_char_level.txt


Evaluating model on test samples: 2700
Loading weights: 100%|██████████| 174/174 [00:00<00:00, 1609.77it/s, Materializing param=model.layers.27.mlp_norm.weight]

Task type: Summary

Evaluating model on 900 samples

---- Example-Level Evaluation ----

Detailed Example-Level Classification Report:
              precision    recall  f1-score   support

   Supported     0.8526    0.9555    0.9011       696
Hallucinated     0.7417    0.4363    0.5494       204

    accuracy                         0.8378       900
   macro avg     0.7971    0.6959    0.7252       900
weighted avg     0.8274    0.8378    0.8214       900


Evaluation Results:

Hallucination Detection (Class 1):
  Precision: 0.7417
  Recall: 0.4363
  F1: 0.5494

Supported Content (Class 0):
  Precision: 0.8526
  Recall: 0.9555
  F1: 0.9011

AUROC: 0.8236

Task type: Data2txt

Evaluating model on 900 samples

---- Example-Level Evaluation ----

Detailed Example-Level Classification Report:
              precision    recall  f

In [5]:
!cd /kaggle/working && zip -qr lettucedetect_full_large.zip lettucedetect_full_large eval_full_*.txt
from huggingface_hub import HfApi
api = HfApi(token=os.environ["HF_TOKEN"])
api.upload_file(path_or_fileobj="/kaggle/working/eval_full_example_level.txt",
                path_in_repo="eval_full_example_level.txt", repo_id="anfs4554/lettucedetect-full-large")
api.upload_file(path_or_fileobj="/kaggle/working/eval_full_char_level.txt",
                path_in_repo="eval_full_char_level.txt", repo_id="anfs4554/lettucedetect-full-large")

CommitInfo(commit_url='https://huggingface.co/anfs4554/lettucedetect-full-large/commit/dd03e816f864e1382f965e1a2bccba70c76dacb4', commit_message='Upload eval_full_char_level.txt with huggingface_hub', commit_description='', oid='dd03e816f864e1382f965e1a2bccba70c76dacb4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/anfs4554/lettucedetect-full-large', endpoint='https://huggingface.co', repo_type='model', repo_id='anfs4554/lettucedetect-full-large'), pr_revision=None, pr_num=None)